# Proyección de Nanopartículas

En esta notebook proyectamos un conjunto de nanopartículas en el espacio tridimensional de acuerdo a la composición de su corona proteica. Mientras más cercanas en el espacio estén dos nanopartículas más similar es su composición.

## La Proyección

Primero instalamos las librerías necesarias.

In [1]:
!pip install umap-learn[plot]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 45.1 MB/s eta 0:00:00


Importamos los recursos desde Github

In [2]:
!git clone https://github.com/matizzat/NanoMed
%cd NanoMed/clustering/

Cloning into 'NanoMed'...
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 88 (delta 41), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (88/88), 12.46 MiB | 3.68 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/content/NanoMed/clustering


Importamos las librerias necesarias.

In [3]:
from dataset import targets, np_without_modification

import plotly.express as px
import plotly.io as pio
import sklearn.datasets
import pandas as pd
import numpy as np
import umap.plot
import umap

/usr/local/lib/python3.12/dist-packages/numba/np/ufunc/dufunc.py:343: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
  warnings.warn(msg, errors.NumbaWarning)
/usr/local/lib/python3.12/dist-packages/numba/np/ufunc/dufunc.py:343: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
  warnings.warn(msg, errors.NumbaWarning)
/usr/local/lib/python3.12/dist-packages/numba/np/ufunc/dufunc.py:343: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
  warnings.warn(msg, errors.NumbaWarning)


Abrimos el lote de datos y mostramos la distribución de frecuencias del atributo `np_without_modification`. Observamos que la mayoría de tipos de nanopartícula tiene una frecuencia menor a 10 instancias y que los tipos más frecuente son SiO2 y Au.

In [4]:
df = pd.read_csv('individual_proteins_dataset.csv')

frequency_distribution = df['np_without_modification'].value_counts().reset_index()
frequency_distribution.columns = ['Nanoparticle Type', 'Frequency']

fig = px.bar(
    frequency_distribution,
    x='Nanoparticle Type',
    y='Frequency',
    title='Frequency Distribution of Nanoparticle Types (np_without_modification)',
    labels={'Nanoparticle Type': 'Type of Nanoparticle', 'Frequency': 'Count'}
)

fig.show()

In [5]:
df['rpa_sum'] = df[targets].sum(axis=1)
print("DataFrame with 'rpa_sum' column added:")
print(df[['np_without_modification', 'rpa_sum']].head())

DataFrame with 'rpa_sum' column added:
  np_without_modification    rpa_sum
0                      PS  99.264147
1                      PS  97.627147
2                      PS  98.874586
3                      PS  98.714233
4                      PS  97.593276


In [6]:
print("First 30 'rpa_sum' values from df:")
display(df['rpa_sum'].head(30))

First 30 'rpa_sum' values from df:


,rpa_sum
0,99.264147
1,97.627147
2,98.874586
3,98.714233
4,97.593276
5,68.240000
6,72.290000
7,65.610000
8,71.380000
9,96.238793


Vamos a imprimir algunas estadísticas importantes respecto a la suma de los valores RPA.

In [7]:
rpa_sum_min = df['rpa_sum'].min()
rpa_sum_max = df['rpa_sum'].max()
rpa_sum_mean = df['rpa_sum'].mean()
rpa_sum_std = df['rpa_sum'].std()

print(f"Minimum rpa_sum: {rpa_sum_min:.2f}")
print(f"Maximum rpa_sum: {rpa_sum_max:.2f}")
print(f"Average rpa_sum: {rpa_sum_mean:.2f}")
print(f"Standard Deviation of rpa_sum: {rpa_sum_std:.2f}")

Minimum rpa_sum: 3.67
Maximum rpa_sum: 100.02
Average rpa_sum: 82.56
Standard Deviation of rpa_sum: 20.93


In [8]:
df_normalized_rpa = df[targets].div(df['rpa_sum'], axis=0)

print("DataFrame with protein RPA values normalized by row's rpa_sum (first 5 rows):")
print(df_normalized_rpa.head())

row_sum_check = df[targets].sum(axis=1)

print((row_sum_check - df['rpa_sum']).abs().describe())

DataFrame with protein RPA values normalized by row's rpa_sum (first 5 rows):
     P01871    P01024    P02647    P02649    P02768    P04004    P01834  \
0  0.006319  0.003305  0.009321  0.001439  0.285002  0.108619  0.027595   
1  0.015829  0.009218  0.175343  0.013519  0.027536  0.058053  0.025306   
2  0.023790  0.010105  0.124213  0.006907  0.004333  0.039722  0.026676   
3  0.013298  0.003770  0.009661  0.004791  0.000895  0.015217  0.011528   
4  0.014281  0.003581  0.003477  0.003461  0.000815  0.008839  0.011104   

     P10909    P02652    P00734  ...  P12814  P61224  Q13201  P67936  P0DOY3  \
0  0.033208  0.000000  0.000211  ...     0.0     0.0     0.0     0.0     0.0   
1  0.347989  0.004067  0.000000  ...     0.0     0.0     0.0     0.0     0.0   
2  0.528948  0.000537  0.000000  ...     0.0     0.0     0.0     0.0     0.0   
3  0.777719  0.002678  0.000000  ...     0.0     0.0     0.0     0.0     0.0   
4  0.790239  0.003859  0.000000  ...     0.0     0.0     0.0     0.0   

Antes de realizar la proyección,filtramos del lote de datos original aquellas partículas cuyo tipo tiene más de 100 instancias.

In [9]:
high_frequency_nanoparticles = frequency_distribution[frequency_distribution['Frequency'] >=100]['Nanoparticle Type'].tolist()
df_filtered_high_frequency = df[df['np_without_modification'].isin(high_frequency_nanoparticles)]

Realizamos la proyección con el algoritmo UMAP de las nanopartículas con los tipos más frecuentes de acuerdo a su composición proteica y utilizando la distancia coseno.

In [10]:
attribute='np_without_modification'
metric='cosine'

pio.renderers.default = "colab"

# Corrected line: Use fit_transform() to get the projected data
mapper_transformed_data = umap.UMAP(n_components=3, n_neighbors=80, metric=metric, random_state=42).fit_transform(df_filtered_high_frequency[targets])

plot_df = pd.DataFrame(mapper_transformed_data, columns=['x', 'y', 'z'])
plot_df['label'] = df_filtered_high_frequency[attribute]
plot_df['index'] = np.arange(len(df_filtered_high_frequency))

fig = px.scatter_3d(
    plot_df,
    x='x',
    y='y',
    z='z',
    color='label',
    hover_data=['index', 'label'],
    title="UMAP 3D Projection",
    opacity=0.7,
    size_max=2
)

#Update marker size for better visibility
fig.update_traces(marker=dict(size=2))

fig.show()

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



A simple vista se observan muchos clusters pequeños y muy apretados. Ambos tipos de nanopartículas tienden a distribuirse por todo el espacio tridimensional pero los clusters son monocromáticos.